# 25 — Veteran Homeless by CoC

Examines veteran homelessness across major cities/CoCs:
- Which CoCs have the most veteran homeless (absolute count)
- Veteran share of total homeless — in which cities are veterans most over-represented among the homeless?


In [1]:
from pathlib import Path
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from scipy import stats

ROOT = Path().resolve().parent
df = pd.read_csv(ROOT / "data" / "processed" / "merged_city_data.csv")
df = df.dropna(subset=["homeless_rate_per_10k", "veteran_rate_per_10k", "veteran_homeless"])
df = df[df["veteran_homeless"] > 0]
print(f"Loaded {len(df)} CoCs with veteran data")


Loaded 101 CoCs with veteran data


In [2]:
df["veteran_share_pct"] = (df["veteran_homeless"] / df["total_homeless"] * 100).round(2)
top25 = df.nlargest(25, "veteran_homeless").copy()

fig1 = px.bar(
    top25.sort_values("veteran_share_pct", ascending=True),
    x="veteran_share_pct", y="city_name", orientation="h",
    color="veteran_rate_per_10k",
    color_continuous_scale="Blues",
    title="Veteran Share of Total Homeless Population by CoC (top 25 by veteran count)<br>"
          "<sup>% of homeless who are veterans — national veteran population share is ~7-8%</sup>",
    labels={"veteran_share_pct": "Veterans as % of all homeless", "city_name": ""},
    hover_name="coc_name",
    hover_data={"veteran_homeless": ":,", "total_homeless": ":,", "veteran_rate_per_10k": ":.2f"},
)
fig1.add_vline(x=8.0, line_dash="dash", line_color="grey", annotation_text="~8% national veteran pop. share")
fig1.update_layout(height=700)
fig1.show()


In [3]:
top20 = df.nlargest(20, "veteran_homeless")
fig2 = px.bar(
    top20.sort_values("veteran_homeless", ascending=True),
    x="veteran_homeless", y="city_name", orientation="h",
    color="veteran_rate_per_10k", color_continuous_scale="Blues",
    title="Top 20 CoCs by Veteran Homeless Count (HUD 2023)",
    labels={"veteran_homeless": "Veteran Homeless Count", "city_name": ""},
    hover_name="coc_name",
)
fig2.show()


In [4]:
print(f"National veteran share of homeless: {df["veteran_share_pct"].mean():.1f}%")
print(f"Highest veteran share: {df.nlargest(5, "veteran_share_pct")[["city_name", "state_postal", "veteran_share_pct", "veteran_homeless"]].to_string(index=False)}")


National veteran share of homeless: 9.0%
Highest veteran share:        city_name state_postal  veteran_share_pct  veteran_homeless
          Austin           TX              11.52               212
        Portland           OR              11.31               712
Colorado Springs           CO              11.18               172
      Fort Worth           TX              10.90               172
         Houston           TX              10.80               512
